In [ ]:
import os
import gc
import scanpy as sc
import pandas as pd
import torch
import scvi

from config import DEFAULT_REFERENCE_CSV, ANALYSIS_RESULTS_DIR
from data_processing import run_cell2location_training
from utils import print_header

# ============================
# 运行配置
# ============================
# 可选: "cpu_full", "gpu_export", "gpu_max"
RUN_PRESET = "cpu_full"

presets = {
    "cpu_full": {
        "use_gpu": False,
        "max_epochs": 20000,
        "batch_size": None,
        "train_size": 1,
        "export_posterior": False,
    },
    "gpu_export": {
        "use_gpu": True,
        "max_epochs": 20000,
        "batch_size": 256,
        "train_size": 1,
        "export_posterior": True,
        "export_batch_size": 512,
    },
    "gpu_max": {
        "use_gpu": True,
        "max_epochs": 10000,
        "batch_size": 2300,
        "train_size": 1,
        "export_posterior": False,
    },
}

cfg = presets[RUN_PRESET]

if not cfg["use_gpu"]:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

scvi.settings.seed = 42
print_header(f"Cell2location preset: {RUN_PRESET}")
print(f"GPU 可用: {torch.cuda.is_available()}")

# ============================
# 输入配置
# ============================
ref_path = str(DEFAULT_REFERENCE_CSV)
ref_df_raw = pd.read_csv(ref_path, index_col=0)

tasks = [
    {"bin": 50, "file": "bin50_raw.h5ad", "n_cells": 1},
    {"bin": 100, "file": "bin100_raw.h5ad", "n_cells": 3},
    {"bin": 150, "file": "bin150_raw.h5ad", "n_cells": 7},
]

# ============================
# 训练主逻辑
# ============================
for task in tasks:
    bin_size = task["bin"]
    spatial_file = task["file"]
    n_cells_mean = task["n_cells"]

    run_name = f"c2l_model_bin{bin_size}"
    save_dir = ANALYSIS_RESULTS_DIR / run_name

    print_header(f"开始处理 Bin {bin_size}")

    if not os.path.exists(spatial_file):
        print(f"❌ 找不到文件 {spatial_file}，跳过。")
        continue

    try:
        adata = sc.read_h5ad(spatial_file)
        adata.var_names_make_unique()
        if "counts" not in adata.layers:
            adata.layers["counts"] = adata.X.copy()

        mod, posterior = run_cell2location_training(
            adata=adata,
            ref_df=ref_df_raw,
            n_cells_mean=n_cells_mean,
            max_epochs=cfg["max_epochs"],
            batch_size=cfg["batch_size"],
            train_size=cfg["train_size"],
            detection_alpha=20,
            accelerator="gpu" if cfg["use_gpu"] else None,
            save_dir=save_dir,
            export_posterior=cfg.get("export_posterior", False),
            export_batch_size=cfg.get("export_batch_size"),
            use_gpu_export=cfg.get("use_gpu", False),
        )

        if posterior is not None:
            mod.adata.obsm["q05_cell_abundance_w_sf"] = posterior["q05_cell_abundance_w_sf"]
            mod.adata.obsm["means_cell_abundance_w_sf"] = posterior["means_cell_abundance_w_sf"]

            result_path = ANALYSIS_RESULTS_DIR / f"results_bin{bin_size}.h5ad"
            mod.adata.write(result_path)
            print(f"💾 结果已保存: {result_path}")

        print(f"🏆 Bin {bin_size} 训练完成。")

    except Exception as e:
        print(f"❌ 错误: {e}")
        import traceback
        traceback.print_exc()

    finally:
        if "mod" in locals():
            del mod
        if "adata" in locals():
            del adata
        gc.collect()
        torch.cuda.empty_cache()
        print("   内存已释放。")
